# HairConsultant — face shape & hair type classifier training

Trains the two on-device classifiers for the HairConsultant Android app:

**Part A — face shape** (Heart / Oval / Round / Square), from three sources:
https://www.kaggle.com/datasets/niten19/face-shape-dataset (Heart/Oblong/Oval/Round/Square),
https://www.kaggle.com/datasets/lucifierx/face-shape-classification (the dataset behind
https://www.kaggle.com/code/lucifierx/face-shape-finder), and
https://huggingface.co/datasets/bkprocovid19/face_shape (Heart/Oblong/Oval/Round/Square). Each source's `Oblong`
class is merged into `Oval`, since the app doesn't distinguish them. The app's `Diamond` class was tried against a
diamond folder in lucifierx's dataset, but it only has 12 images — far too few to train on — so Diamond
intentionally stays covered only by the app's existing geometric heuristic (see `FaceShapeTfliteClassifier`'s doc
comment on the Android side).

**Part B — hair type** (Straight / Wavy / Curly), from
https://www.kaggle.com/datasets/kavyasreeb/hair-type-dataset (Straight/Wavy/Curly/Kinky, plus a Dreadlocks
folder). The app doesn't distinguish a separate tightly-coiled texture from curly, so this notebook folds the
source's `Kinky` class into `Curly` rather than training it as its own class — see Part B's section 8 for the
merge. `Dreadlocks` is dropped entirely, same as before: it's a hairstyle, not a hair texture.

Unlike face shape, hair type genuinely needs pixel information — texture is exactly the kind of visual pattern a
CNN is good at — so Part B fine-tunes a MobileNetV2 (ImageNet weights) rather than training on landmark geometry.

Part A **doesn't train on pixels at all**. Face shape is fundamentally geometric — it's what the app's existing
heuristic in `FaceShapeClassifier.kt` already measures from eight MediaPipe face landmarks (`lengthToWidth`,
`jawToCheek`, `foreheadToJaw`, `foreheadToCheek`). An earlier pixel-based CNN version of this notebook kept
producing degenerate on-device results despite looking fine in Colab evaluation, traced back to the training
photos' framing not matching what the app's live camera crop actually looks like at inference time — a
domain-shift problem no amount of extra training could fix. Training a small feedforward network on those same
four numbers instead removes that failure class entirely: whoever computes them — Python here, Kotlin on-device
— is measuring the same landmark geometry, so nothing can drift out of sync between training and inference. The
resulting model is a few KB, not several MB.

(An earlier version of Part A used `mediapipe-model-maker` for the image-CNN approach, which pulls in
`tensorflow-text`/`scann` and reliably fails to install on current Colab Python versions — a long-standing,
widely reported issue. That's moot now since Part A doesn't need image classification at all, but is why the
notebook uses plain Keras rather than that library.)

Because there's no embedded-metadata classifier format this way, the Android app loads `face_shape_classifier.tflite`
with the raw TensorFlow Lite `Interpreter`, feeding it the same four ratio numbers `FaceShapeClassifier.kt`
already computes, and maps output indices back to labels using a **hardcoded array that must match the
alphabetical class order** this notebook trains with (`FaceShapeTfliteClassifier.LABELS` in the Android project —
Part A's section 5 below asserts that order explicitly so a mismatch fails loudly here instead of silently
mislabeling on-device). `hair_type_classifier.tflite` is loaded the same way, against
`HairTypeTfliteClassifier.LABELS` — Part B's training section asserts its class order too.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine and free — Part A's model and dataset are
tiny so it trains fast either way, but Part B's MobileNetV2 fine-tuning is meaningfully faster on GPU. The
landmark-extraction step in Part A's section 4c is the slow part for Part A and runs on CPU regardless).

**You'll need:** a Kaggle API token, used by both parts. On kaggle.com/settings, scroll to **"Legacy API
Credentials"** (not the newer "API → Create New Token", which doesn't reliably download a file) and click
**"Create Legacy API Key"** — that downloads `kaggle.json`. You'll upload that file in the next section; it's used
only inside this Colab session. (Part A's Hugging Face source needs no credentials at all.)

**At the end of each part** you'll download its `.tflite` file — copy `face_shape_classifier.tflite` and/or
`hair_type_classifier.tflite` into `app/src/main/assets/` in the Android project (overwriting the existing file),
and the app will start using whichever one you replaced automatically. You can run just one part if you only want
to retrain one classifier — they don't depend on each other.

## 1. Setup
Nothing to build from source here — `tensorflow` ships with Colab and `kaggle` is preinstalled; this just confirms versions.

In [ ]:
!pip install -q --upgrade kaggle
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 2. Kaggle credentials
Run this cell, then upload the `kaggle.json` you downloaded from kaggle.com/settings (Legacy API Credentials).

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
assert 'kaggle.json' in uploaded, "Please upload the kaggle.json you downloaded from kaggle.com/settings"

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials saved.')

## 3. Download the datasets
Two Kaggle sources for face shape (niten19 + lucifierx, into separate `raw/` folders so their unzips can't
overwrite each other) plus a Hugging Face source in section 3b below. Add more `!kaggle datasets download ...`
lines the same way (own raw folder per source) if you want to fold in yet another dataset — just extend
`FACE_SHAPE_KEYWORDS` and the `find_class_dirs` calls in section 4 so the new source's folder names get picked
up too.

In [ ]:
!kaggle datasets download -d niten19/face-shape-dataset -p /content/raw/face_shape --unzip
!kaggle datasets download -d lucifierx/face-shape-classification -p /content/raw/face_shape_extra --unzip

## 3b. Download a Hugging Face face-shape source too (bonus data, no Diamond)
`bkprocovid19/face_shape` on Hugging Face is a separate, independently-hosted 5,000-image dataset — no Kaggle
credentials needed. Its model card claims Oval/Round/Square/Heart/**Diamond**, but running this actually prints
Heart/Oblong/Oval/Round/Square (confirmed by running it — the model card was wrong). So this source does **not**
help with Diamond; it's included anyway as bonus data for the other four classes, exported into the same `raw/`
folder-per-class layout as the Kaggle sources so section 4 folds it in the same way.

In [ ]:
!pip install -q datasets

import os
from datasets import load_dataset
from PIL import ImageFile, UnidentifiedImageError

# A handful of images in this dataset are truncated at the source; without this, PIL raises
# instead of loading the bytes it actually received.
ImageFile.LOAD_TRUNCATED_IMAGES = True

hf_root = '/content/raw/face_shape_hf'
os.makedirs(hf_root, exist_ok=True)

hf_ds = load_dataset('bkprocovid19/face_shape', split='train')
hf_label_names = hf_ds.features['label'].names
print('bkprocovid19/face_shape actual classes:', hf_label_names)
if not any('diamond' in name.lower() for name in hf_label_names):
    print('NOTE: this dataset has no Diamond class (its model card was wrong about that) — it '
          'only contributes bonus images for the other face-shape classes.')

for name in hf_label_names:
    os.makedirs(os.path.join(hf_root, name), exist_ok=True)

hf_counts = {name: 0 for name in hf_label_names}
skipped = 0
# Indexing one item at a time (rather than `for example in hf_ds`) so a decode failure on item i
# raises here, inside this try/except, instead of inside the dataset library's own hidden
# iteration/formatting step where a loop-body try/except can't catch it.
for i in range(len(hf_ds)):
    try:
        example = hf_ds[i]
        img = example['image'].convert('RGB')
        img.save(os.path.join(hf_root, hf_label_names[example['label']], f'{i:05d}.jpg'), format='JPEG')
    except (OSError, UnidentifiedImageError):
        skipped += 1
        continue
    hf_counts[hf_label_names[example['label']]] += 1
print('bkprocovid19/face_shape per-class counts:', hf_counts, f'(skipped {skipped} corrupt image(s))')

## 4. Normalize the three face-shape sources into clean class folders
Kaggle/Hugging Face layouts vary (train/test splits, casing, etc.), so this walks the extracted tree and matches
directory names against known class keywords — print the matched paths below and sanity-check them before
training.

In [ ]:
import os, re, shutil

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def _normalize(name):
    """Lowercases and strips everything but letters, so 'Heart_Shape', 'heart-shaped' and
    'HeartFaces' all normalize to something containing 'heart'."""
    return re.sub(r'[^a-z]', '', name.lower())

def find_class_dirs(root, class_keywords):
    """Walks root; any directory whose normalized name *contains* one of a class's keywords,
    and directly contains image files, is added as a source directory for that class.
    Uses substring containment (not exact match) so folder-naming variants across different
    dataset authors (e.g. 'Heart_Shape' vs 'heart') still get picked up — an exact-match version
    of this silently dropped entire classes from a second dataset whose folders weren't named
    identically to the first, skewing training data toward whichever class *did* match and
    training a model that just always predicted that one class."""
    found = {cls: [] for cls in class_keywords}
    for dirpath, _dirnames, filenames in os.walk(root):
        name_norm = _normalize(os.path.basename(dirpath))
        imgs = [f for f in filenames if f.lower().endswith(IMG_EXTS)]
        if not imgs:
            continue
        for cls, patterns in class_keywords.items():
            if any(_normalize(p) in name_norm for p in patterns):
                found[cls].append(dirpath)
                break  # a folder belongs to at most one class
    return found

def merge_into(target_root, class_dir_map):
    os.makedirs(target_root, exist_ok=True)
    counts = {}
    for cls, dirs in class_dir_map.items():
        dest = os.path.join(target_root, cls)
        os.makedirs(dest, exist_ok=True)
        n = 0
        for d in dirs:
            for f in os.listdir(d):
                if f.lower().endswith(IMG_EXTS):
                    shutil.copyfile(os.path.join(d, f), os.path.join(dest, f'{n:05d}_{f}'))
                    n += 1
        counts[cls] = n
    return counts

In [ ]:
# Class folder names match the app's Kotlin enum names exactly (FaceShape.HEART, .OVAL, ...)
# so the Android side can map a predicted label straight back to the enum with no lookup table.
# DIAMOND is intentionally not a key here: lucifierx's dataset has a diamond folder, but it only
# has 12 images in it — nowhere near enough to train a class on. Diamond stays covered by the
# app's geometric heuristic only (see FaceShapeTfliteClassifier's doc comment).
FACE_SHAPE_KEYWORDS = {
    'HEART': ['heart', 'hearts'],
    'OVAL': ['oval', 'ovals', 'oblong', 'oblongs'],
    'ROUND': ['round', 'rounds'],
    'SQUARE': ['square', 'squares'],
}

def merge_keyword_maps(*maps):
    combined = {}
    for m in maps:
        for cls, dirs in m.items():
            combined.setdefault(cls, []).extend(dirs)
    return combined

# Three independent face-shape sources, downloaded into separate raw/ folders (section 3/3b) so
# their unzips/exports can't collide with each other, then combined here before copying into
# /content/data.
face_dirs_niten19 = find_class_dirs('/content/raw/face_shape', FACE_SHAPE_KEYWORDS)
face_dirs_lucifierx = find_class_dirs('/content/raw/face_shape_extra', FACE_SHAPE_KEYWORDS)
face_dirs_hf = find_class_dirs('/content/raw/face_shape_hf', FACE_SHAPE_KEYWORDS)
for source_name, source_dirs in [
    ('niten19', face_dirs_niten19), ('lucifierx', face_dirs_lucifierx), ('hf', face_dirs_hf)
]:
    for cls, dirs in source_dirs.items():
        print(source_name, cls, '->', dirs)

face_dirs = merge_keyword_maps(face_dirs_niten19, face_dirs_lucifierx, face_dirs_hf)
face_counts = merge_into('/content/data/face_shape', face_dirs)
print(face_counts)
assert all(c > 50 for c in face_counts.values()), (
    'A face-shape class has too few images — check the printed paths above; the dataset layout '
    'may not match FACE_SHAPE_KEYWORDS.'
)

## 4b. Drop images TensorFlow can't decode
Kaggle/Hugging Face image sets almost always contain a handful of truncated files or files whose real format
doesn't match their extension. PIL is more lenient about minor corruption than TensorFlow's own JPEG decoder is,
so a file can pass a PIL-based check and still crash `image_dataset_from_directory` mid-epoch (`jpeg::Uncompress
failed`) — which is exactly what happens if you hit that error even after this cell has run. To guarantee that
can't happen, this validates every file with `tf.io.decode_image` itself — the same decoder training uses — and
deletes anything it rejects, then re-prints counts so you can confirm each class still has enough images left.

In [ ]:
import tensorflow as tf

def clean_images(root):
    removed = 0
    for dirpath, _dirs, filenames in os.walk(root):
        for fname in filenames:
            path = os.path.join(dirpath, fname)
            ok = False
            try:
                data = tf.io.read_file(path)
                img = tf.io.decode_image(data, channels=3, expand_animations=False)
                ok = int(img.shape[0]) > 0 and int(img.shape[1]) > 0
            except Exception:
                ok = False
            if not ok:
                os.remove(path)
                removed += 1
    return removed

removed = clean_images('/content/data/face_shape')
counts = {d: len(os.listdir(os.path.join('/content/data/face_shape', d))) for d in sorted(os.listdir('/content/data/face_shape'))}
print(f'face_shape: removed {removed} undecodable file(s); remaining counts {counts}')
assert all(c > 50 for c in counts.values()), 'face_shape has a class with too few images left after cleanup.'

## 4c. Extract landmark-ratio features instead of training on pixels
Face shape is fundamentally geometric — that's what the app's existing heuristic in `FaceShapeClassifier.kt`
already measures (`lengthToWidth`, `jawToCheek`, `foreheadToJaw`, `foreheadToCheek`, from eight MediaPipe
landmarks). A pixel-based CNN version of this classifier kept producing degenerate results in the app despite
looking fine in isolated Colab evaluation — traced back to the training photos' framing not matching what the
app's live camera crop actually looks like at inference time (a domain-shift problem no amount of extra training
epochs could fix).

Training on these four numbers instead of raw pixels removes that whole failure class: whoever computes them —
Python here, Kotlin on-device — is measuring the same landmark geometry, so there's nothing that can drift out of
sync between training and inference. This runs the same MediaPipe Face Landmarker the app uses on every training
image and computes the same four ratios `FaceShapeClassifier.kt`'s heuristic does, building a small tabular
dataset instead of a folder of cropped images.

In [ ]:
!pip install -q mediapipe
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
assert os.path.getsize('/content/face_landmarker.task') > 1_000_000, (
    'face_landmarker.task did not download correctly (file too small/empty). Check '
    'https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker for the current model '
    'download URL and update the wget line above if Google has moved it.'
)

import math
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions
from mediapipe.tasks.python.vision import FaceLandmarker, FaceLandmarkerOptions, RunningMode
from PIL import Image, UnidentifiedImageError

# Same landmark indices and ratio formulas as FaceShapeClassifier.kt's heuristic — keep these in
# sync if the Kotlin side ever changes them.
FOREHEAD_TOP, CHIN = 10, 152
LEFT_CHEEK, RIGHT_CHEEK = 234, 454
LEFT_JAW, RIGHT_JAW = 172, 397
LEFT_TEMPLE, RIGHT_TEMPLE = 127, 356

landmarker = FaceLandmarker.create_from_options(FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='/content/face_landmarker.task'),
    running_mode=RunningMode.IMAGE,
    num_faces=1,
))

def _dist(landmarks, i, j, w, h):
    xi, yi = landmarks[i].x, landmarks[i].y
    xj, yj = landmarks[j].x, landmarks[j].y
    return math.hypot((xi - xj) * w, (yi - yj) * h)

def extract_metrics(image_path):
    """Returns [lengthToWidth, jawToCheek, foreheadToJaw, foreheadToCheek], or None if no face /
    degenerate geometry — mirrors FaceShapeClassifier.classify()'s early-outs in Kotlin."""
    try:
        img = Image.open(image_path).convert('RGB')
    except (OSError, UnidentifiedImageError):
        return None
    w, h = img.size
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(img))
    try:
        result = landmarker.detect(mp_image)
    except Exception:
        return None
    if not result.face_landmarks:
        return None
    landmarks = result.face_landmarks[0]

    length = _dist(landmarks, FOREHEAD_TOP, CHIN, w, h)
    cheek = _dist(landmarks, LEFT_CHEEK, RIGHT_CHEEK, w, h)
    jaw = _dist(landmarks, LEFT_JAW, RIGHT_JAW, w, h)
    temple = _dist(landmarks, LEFT_TEMPLE, RIGHT_TEMPLE, w, h)
    if length < 8 or cheek < 8 or jaw < 8 or temple < 8:
        return None
    return [length / cheek, jaw / cheek, temple / jaw, temple / cheek]

FACE_SHAPE_CLASSES = ['HEART', 'OVAL', 'ROUND', 'SQUARE']
features, labels = [], []
dropped = 0
for class_idx, cls in enumerate(FACE_SHAPE_CLASSES):
    src_dir = os.path.join('/content/data/face_shape', cls)
    for fname in os.listdir(src_dir):
        metrics = extract_metrics(os.path.join(src_dir, fname))
        if metrics is None:
            dropped += 1
            continue
        features.append(metrics)
        labels.append(class_idx)

X = np.array(features, dtype=np.float32)
y = np.array(labels, dtype=np.int64)
print(f'Extracted features for {len(X)} images; dropped {dropped} with no detected face.')
counts = {cls: int((y == i).sum()) for i, cls in enumerate(FACE_SHAPE_CLASSES)}
print('Per-class counts:', counts)
assert all(c > 50 for c in counts.values()), (
    'A class has too few images with detected faces — check the dropped count above.'
)

## 5. Train and export the face-shape classifier
A small feedforward network on the 4 landmark ratios — plain `sklearn.train_test_split(..., stratify=y)` for the
train/val/test split (no `image_dataset_from_directory` shuffle/validation_split footguns to worry about here),
a `Normalization` layer baked into the model so the Android side just feeds raw ratios, and no quantization (tiny
model already, and quantization is what broke on-device loading before).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
print(f'train={len(X_train)} val={len(X_val)} test={len(X_test)}')

class_counts = np.bincount(y_train, minlength=len(FACE_SHAPE_CLASSES))
total = class_counts.sum()
class_weight = {i: total / (len(FACE_SHAPE_CLASSES) * class_counts[i]) for i in range(len(FACE_SHAPE_CLASSES))}
print('Class weights (from training split):', dict(zip(FACE_SHAPE_CLASSES, [round(class_weight[i], 2) for i in range(len(FACE_SHAPE_CLASSES))])))

normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(X_train)

face_model = tf.keras.Sequential([
    tf.keras.Input(shape=(4,)),
    normalizer,
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(FACE_SHAPE_CLASSES), activation='softmax'),
])
face_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True)
face_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=150, batch_size=32,
                callbacks=[early_stop], class_weight=class_weight, verbose=2)

face_test_loss, face_test_acc = face_model.evaluate(X_test, y_test, verbose=0)
print(f'face_shape_classifier.tflite: held-out test accuracy {face_test_acc:.3f}')

y_pred = np.argmax(face_model.predict(X_test, verbose=0), axis=1)
true_counts = np.bincount(y_test, minlength=len(FACE_SHAPE_CLASSES))
print('Test set true-label counts:', dict(zip(FACE_SHAPE_CLASSES, true_counts)))
assert (true_counts > 0).all(), 'The held-out test set is missing at least one class entirely.'

cm = tf.math.confusion_matrix(y_test, y_pred, num_classes=len(FACE_SHAPE_CLASSES)).numpy()
print('Confusion matrix (rows=true, cols=predicted), class order', FACE_SHAPE_CLASSES)
print(cm)
per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
print('Per-class recall:', dict(zip(FACE_SHAPE_CLASSES, per_class_recall.round(3))))
if np.any(per_class_recall < 0.15):
    print('WARNING: at least one class is almost never being predicted correctly — check the '
          'per-class counts and confusion matrix above before deploying this .tflite.')

# No quantization: this model is tiny (a few dense layers on 4 numbers) so there's no size problem
# to solve, and quantization is exactly what made the earlier pixel-based model's FULLY_CONNECTED
# op version too new for the app's bundled TFLite runtime to load at all.
converter = tf.lite.TFLiteConverter.from_keras_model(face_model)
tflite_model = converter.convert()
with open('face_shape_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'Exported face_shape_classifier.tflite ({len(tflite_model)/1e3:.1f} KB)')

## 6. Download the face-shape model
Copy the downloaded file into `app/src/main/assets/` in the HairConsultant Android project (keep the exact file
name — the app looks for it), overwriting the existing `face_shape_classifier.tflite`. `hair_type_classifier.tflite`
is retrained separately in Part B below — this only produces the face-shape model.

In [ ]:
from google.colab import files

files.download('face_shape_classifier.tflite')

# Part B — Hair type classifier

## 7. Download the hair-type dataset
[kavyasreeb/hair-type-dataset](https://www.kaggle.com/datasets/kavyasreeb/hair-type-dataset) on Kaggle —
Straight/Wavy/Curly/Kinky, plus a Dreadlocks folder. Reuses the `kaggle.json` credentials uploaded in section 2,
so run that cell first if you're only doing Part B in a fresh session.

In [ ]:
!kaggle datasets download -d kavyasreeb/hair-type-dataset -p /content/raw/hair_type --unzip

## 8. Normalize into class folders, merging Kinky into Curly
Reuses `find_class_dirs`/`merge_into` from section 4 — same substring-matching approach, now applied to the
hair-type source. The source's `Kinky` folder is given as a pattern for the `CURLY` class rather than its own
class, so its images land in `/content/data/hair_type/CURLY` alongside the source's own `Curly` folder — this is
the actual "join the coily dataset into curly" step. `Dreadlocks` has no entry in `HAIR_TYPE_KEYWORDS` at all, so
`find_class_dirs` never matches it and none of its images get copied anywhere: it's a hairstyle, not a texture,
so the app has nothing to map it to.

In [ ]:
# Class folder names match the app's Kotlin enum names exactly (HairTexture.STRAIGHT, .WAVY,
# .CURLY) so the Android side can map a predicted label straight back to the enum with no lookup
# table. Straight/Wavy/Curly are alphabetical, matching HairTypeTfliteClassifier.LABELS.
#
# 'kinky' and 'coily' are both listed as CURLY patterns: the source dataset's class is named
# Kinky, but 'coily' is included too in case a re-run ever points at a differently-named source —
# same defensive style as FACE_SHAPE_KEYWORDS's plural variants above. The app doesn't distinguish
# a separate tightly-coiled texture from curly, so these join the source's own Curly folder rather
# than becoming their own class.
#
# DREADLOCKS is intentionally not a key here — it's a hairstyle, not a hair texture, so its images
# are simply never matched and never copied into /content/data/hair_type.
HAIR_TYPE_KEYWORDS = {
    'STRAIGHT': ['straight'],
    'WAVY': ['wavy', 'wave', 'waves'],
    'CURLY': ['curly', 'curls', 'kinky', 'coily'],
}

hair_dirs = find_class_dirs('/content/raw/hair_type', HAIR_TYPE_KEYWORDS)
for cls, dirs in hair_dirs.items():
    print(cls, '->', dirs)

hair_counts = merge_into('/content/data/hair_type', hair_dirs)
print(hair_counts)
assert all(c > 50 for c in hair_counts.values()), (
    'A hair-type class has too few images — check the printed paths above; the dataset layout '
    'may not match HAIR_TYPE_KEYWORDS.'
)

## 8b. Drop images TensorFlow can't decode
Reuses `clean_images` from section 4b for the same reason it was needed there — a file can pass a lenient check
and still crash `image_dataset_from_directory` mid-epoch.

In [ ]:
removed = clean_images('/content/data/hair_type')
counts = {d: len(os.listdir(os.path.join('/content/data/hair_type', d))) for d in sorted(os.listdir('/content/data/hair_type'))}
print(f'hair_type: removed {removed} undecodable file(s); remaining counts {counts}')
assert all(c > 50 for c in counts.values()), 'hair_type has a class with too few images left after cleanup.'

## 9. Build train/val/test datasets
Unlike face shape, hair texture genuinely lives in the pixels — curl pattern is exactly the kind of visual
texture a CNN reads well, and it's what the app's own pixel-heuristic fallback (`HairAppearanceClassifier.kt`)
already approximates by measuring edge sign-changes across the hair region. So this trains an actual image
classifier instead of hand-computed ratios.

`image_dataset_from_directory` gives an 80/20 train/temp split directly (with a fixed `seed` so it's
reproducible); the 20% temp portion is then halved into val/test by batch count, since
`image_dataset_from_directory` itself only supports a single validation split. Images are resized to 224x224 —
MobileNetV2's expected input size, and the same size `HairTypeTfliteClassifier.bitmapToInputBuffer` resizes to on
the Android side — and preprocessed with `mobilenet_v2.preprocess_input`, which scales pixels to [-1, 1] exactly
like `bitmapToInputBuffer` already does per-channel, so nothing needs to change there.

In [ ]:
HAIR_TYPE_CLASSES = ['CURLY', 'STRAIGHT', 'WAVY']
IMG_SIZE = 224
BATCH_SIZE = 32

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/data/hair_type', validation_split=0.2, subset='training', seed=42,
    label_mode='int', class_names=HAIR_TYPE_CLASSES, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
)
raw_temp_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/data/hair_type', validation_split=0.2, subset='validation', seed=42,
    label_mode='int', class_names=HAIR_TYPE_CLASSES, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
)
# Must match HairTypeTfliteClassifier.LABELS's order exactly, or the exported model's output
# indices will silently mean something different on-device than they do here.
assert raw_train_ds.class_names == HAIR_TYPE_CLASSES and raw_temp_ds.class_names == HAIR_TYPE_CLASSES, (
    'image_dataset_from_directory class order does not match HAIR_TYPE_CLASSES — check folder '
    'names under /content/data/hair_type.'
)

temp_batches = tf.data.experimental.cardinality(raw_temp_ds)
raw_val_ds = raw_temp_ds.take(temp_batches // 2)
raw_test_ds = raw_temp_ds.skip(temp_batches // 2)

AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds = raw_train_ds.map(lambda x, y: (preprocess(x), y)).prefetch(AUTOTUNE)
val_ds = raw_val_ds.map(lambda x, y: (preprocess(x), y)).prefetch(AUTOTUNE)
test_ds = raw_test_ds.map(lambda x, y: (preprocess(x), y)).prefetch(AUTOTUNE)

print(f'train batches={tf.data.experimental.cardinality(train_ds).numpy()} '
      f'val batches={tf.data.experimental.cardinality(val_ds).numpy()} '
      f'test batches={tf.data.experimental.cardinality(test_ds).numpy()}')

## 10. Train and export the hair-type classifier
MobileNetV2 transfer learning in the standard two-stage recipe: first train a small classification head on top
of the frozen, ImageNet-pretrained base (light augmentation only, since the base weights don't move yet), then
unfreeze the base model's last 30 layers and continue training at a much lower learning rate to fine-tune those
features for hair texture specifically. No quantization, matching Part A's model and for the same reason noted
there — quantization is what broke on-device TFLite loading before.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(HAIR_TYPE_CLASSES), activation='softmax')(x)
hair_model = tf.keras.Model(inputs, outputs)

hair_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
hair_model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[early_stop])

# Stage 2: unfreeze the base model's last 30 layers and fine-tune at a much lower learning rate,
# so the pretrained ImageNet features shift gently toward hair-texture-specific ones instead of
# being clobbered by a high learning rate.
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

hair_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hair_model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop])

In [ ]:
hair_test_loss, hair_test_acc = hair_model.evaluate(test_ds, verbose=0)
print(f'hair_type_classifier.tflite: held-out test accuracy {hair_test_acc:.3f}')

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_pred = np.argmax(hair_model.predict(test_ds, verbose=0), axis=1)
true_counts = np.bincount(y_true, minlength=len(HAIR_TYPE_CLASSES))
print('Test set true-label counts:', dict(zip(HAIR_TYPE_CLASSES, true_counts)))
assert (true_counts > 0).all(), 'The held-out test set is missing at least one class entirely.'

cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=len(HAIR_TYPE_CLASSES)).numpy()
print('Confusion matrix (rows=true, cols=predicted), class order', HAIR_TYPE_CLASSES)
print(cm)
per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
print('Per-class recall:', dict(zip(HAIR_TYPE_CLASSES, per_class_recall.round(3))))
if np.any(per_class_recall < 0.15):
    print('WARNING: at least one class is almost never being predicted correctly — check the '
          'per-class counts and confusion matrix above before deploying this .tflite.')

# No quantization, same reasoning as the face-shape model: quantization is what made an earlier
# model version too new for the app's bundled TFLite runtime to load at all.
converter = tf.lite.TFLiteConverter.from_keras_model(hair_model)
tflite_hair_model = converter.convert()
with open('hair_type_classifier.tflite', 'wb') as f:
    f.write(tflite_hair_model)
print(f'Exported hair_type_classifier.tflite ({len(tflite_hair_model)/1e6:.2f} MB)')

## 11. Download the hair-type model
Copy the downloaded file into `app/src/main/assets/` in the HairConsultant Android project, overwriting the
existing `hair_type_classifier.tflite`.

In [ ]:
from google.colab import files

files.download('hair_type_classifier.tflite')

## Notes
- **Diamond face shape** is intentionally excluded from `FACE_SHAPE_CLASSES` — lucifierx's dataset does have a
  diamond folder, but it only holds 12 images, nowhere near enough to train a class on. The app's
  landmark-geometry heuristic covers Diamond as its only source, and that's a deliberate, permanent design
  choice rather than a stopgap: Diamond is fundamentally a geometric definition (cheekbones clearly the widest
  zone, forehead and jaw both narrower), which the heuristic already captures well.
- **Face-shape accuracy tuning**: this model is tiny and trains in seconds, so the cheap lever is just re-running
  section 5 (it reshuffles the train/val/test split via a different `random_state` if you change it, and
  `EarlyStopping` patience/epochs are easy to bump). If accuracy is still weak, the more likely culprit is the
  *features*, not the model — e.g. adding more geometric ratios beyond the four `FaceShapeClassifier.kt`
  currently computes would need matching changes on the Kotlin side too, not just here.
- Re-running section 4 is safe — it wipes and rebuilds `/content/data/face_shape` from the raw downloads each
  time. Section 4c depends on section 4's output, not on anything from the old pixel-based pipeline.
- **Coily/Kinky hair is not its own class** — this used to be trained as a separate `COILY` class (with the
  source dataset's `Kinky` folder renamed to it), but the app doesn't actually distinguish a fourth tightly-coiled
  texture from curly in its UI, recommendations, or knowledge base, so section 8 now folds `Kinky` straight into
  `CURLY` instead. This was a deliberate simplification, not a data-availability workaround like Diamond above —
  removing it also required updating `HairTexture` (Kotlin enum), `HairTypeTfliteClassifier.LABELS`, the pixel
  heuristic in `HairAppearanceClassifier.kt`, the haircut catalog in `SampleData.kt`, and the chatbot's knowledge
  base — all on the Android side, not in this notebook.
- **Hair-type accuracy tuning**: unlike Part A, Part B trains on real pixels, so accuracy is more sensitive to
  standard CNN levers — more/fewer unfrozen layers in the fine-tuning stage, more epochs, a lower fine-tuning
  learning rate, or stronger/weaker augmentation. Re-running section 8 is safe the same way section 4 is; it
  wipes and rebuilds `/content/data/hair_type` from the raw download each time.